# Stage 10 (repurposed) — General F&O trend-following 1D Supertrend screener - full-universe scale

REPURPOSED from the original pledged-securities-only build (see
`trend_universe.py` and README Stage 10 for the history). The universe is
now the **full F&O stock+index universe** (`get_fno_stock_list()` + the 6
canonical indices in `src.instruments.INDEX_SYMBOLS`), not the 154-symbol
pledgeable-securities list — the strategy this feeds is pure cash equity, no
F&O instrument involved: buy on a fresh 1D Supertrend UP flip, hold, exit on
the opposite DOWN flip.

Supertrend flip detection on the **1D** timeframe (not 4H) - parallel to the
existing credit-spread pipeline, reusing `supertrend.py`/`flip_analysis.py`/
`monthly_range.py`/`summary_stats.py` exactly as-is. Writes to
`exports/trend_1d/` - a separate folder, so this can NEVER overwrite the
existing 4H reports for the many stocks that are in both universes.

No stop-loss: `src/stop_loss_test.py` backtested fixed-%, ATR-multiple, and
worst-day-based candidate stop rules against real 1D flip history and found
none beat holding to the natural DOWN-flip exit — every rule's stopped-out
flips would have averaged a POSITIVE outcome if held (consistent with
`reversion_test.py`'s momentum/continuation finding). Exit is the Supertrend
DOWN flip only, by design.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token_lookup
from src.trend_universe import load_trend_universe, save_trend_universe
from src.trend_1d_pipeline import run_all_instruments

kite = get_kite()
universe = load_trend_universe(kite=kite)
save_trend_universe(fno_symbol_list=universe[universe.security_type == "Stock"]["symbol"].tolist())
symbol_list = universe["symbol"].tolist()
print(f"{len(symbol_list)} instruments to process "
      f"({(universe.security_type=='Stock').sum()} stocks, {(universe.security_type=='Index').sum()} indices)")

token_lookup = build_nse_token_lookup(kite)
token_lookup.update(get_index_token_lookup(kite))

# incremental=True rather than force_refresh=True: process_instrument_incremental
# already does a genuine deep fetch (from deep_from_date=2015) for any symbol with
# no saved data yet, and a cheap catch-up for symbols already covered — same
# complete, correct result as a full force_refresh, without re-pulling ~11 years
# of daily candles for every stock that's already covered from a prior run.
results = run_all_instruments(
    kite, token_lookup, symbol_list,
    from_date=datetime(2015, 1, 1),
    to_date=datetime.now(),
    incremental=True
)
results

Saved 220 symbols (214 stocks, 6 indices) to /Users/hgmac/Documents/3_trading/Quant/config/trend_universe.csv
220 instruments to process (214 stocks, 6 indices)
[1/220] 360ONE — processing...
Fetching 2015-01-01 to 2016-01-31...
Fetching 2016-02-01 to 2017-03-02...
Fetching 2017-03-03 to 2018-04-02...
Fetching 2018-04-03 to 2019-05-03...
Fetching 2019-05-04 to 2020-06-02...
Fetching 2020-06-03 to 2021-07-03...
Fetching 2021-07-04 to 2022-08-03...
Fetching 2022-08-04 to 2023-09-03...
Fetching 2023-09-04 to 2024-10-03...
Fetching 2024-10-04 to 2025-11-03...
Fetching 2025-11-04 to 2026-10-27...
Saved 2018 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/360ONE_full_report.xlsx
[1/220] 360ONE — done
[2/220] ABB — processing...
Fetching 2015-01-01 to 2016-01-31...
Fetching 2016-02-01 to 2017-03-02...
Fetching 2017-03-03 to 2018-04-02...
Fetching 2018-04-03 to 2019-05-03...
Fetching 2019-05-

,symbol,status,error
0,360ONE,SUCCESS,
1,ABB,SUCCESS,
2,ABCAPITAL,SUCCESS,
3,ADANIENSOL,SUCCESS,
4,ADANIENT,SUCCESS,
...,...,...,...
215,BANKNIFTY,SUCCESS,
216,SENSEX,SUCCESS,
217,NIFTYNXT50,SUCCESS,
218,MIDCPNIFTY,SUCCESS,


`from_date=2015` is intentionally much further back than the 4H universe's usual
window - daily candles are ~6x coarser than 4H, so flips are proportionally rarer
per calendar year. A recently-listed stock/ETF will simply return less data (Kite
just gives what exists), no error.

In [2]:
from src.trend_1d_compare import build_comparison_table, save_comparison

comparison = build_comparison_table(symbol_list)
save_comparison(comparison)

Skipping NIFTYFPI — report not found
Skipping WAAREEENER — report not found
Saved comparison of 218 instruments to /Users/hgmac/Documents/3_trading/Quant/exports/TREND_1D_COMPARISON.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/TREND_1D_COMPARISON.xlsx'